# 01 – Recogida de Datos (Data Collection)

**Responsable**: David (Data Analyst)  
**Fecha**: Mayo 2026  
**Objetivo**: Obtener ofertas de empleo del sector datos en España mediante scraping de portales de empleo.

## Fuentes evaluadas

| Portal | Método | Resultado |
|--------|--------|-----------|
| InfoJobs | requests + BeautifulSoup | Bloqueo (CAPTCHA) |
| InfoJobs | Playwright (navegador real) | Bloqueo (Distil Networks) |
| Indeed | requests + BeautifulSoup | Bloqueo (403 Security Check) |
| Indeed | Playwright visible | **Exitoso** |

**Conclusión**: Solo Indeed permite scraping si se utiliza un navegador real no headless.

## Consideraciones éticas y legales
- Se respeta `robots.txt` de Indeed en la medida de lo posible.
- Se utiliza un User-Agent realista y un retardo entre peticiones para no sobrecargar el servidor.
- Los datos extraídos se usan exclusivamente con fines educativos dentro de este proyecto.

## Cómo ejecutar el scraping

La función `scrape_indeed()` ya está definida más abajo y lista para usarse.

**Ejemplo 1 – Búsqueda simple**  
Ejecuta la siguiente celda para obtener ofertas de "analista de datos" en Madrid:

```python
df = await scrape_indeed("analista de datos", "Madrid", headless=False)
df.head()
```

**Ejemplo 2 – Múltiples búsquedas en lote**  
Para buscar varios perfiles a la vez, añade una celda como esta:

```python
busquedas = [
    ("data scientist", "Madrid"),
    ("data engineer", "Barcelona"),
    ("machine learning", "Valencia"),
]
for kw, loc in busquedas:
    await scrape_indeed(keyword, location, headless=False)
```

Todas las ofertas se acumularán automáticamente en  
`data/raw/scraped_jobs.csv`, sin duplicados.
```

Ejecútala para que se renderice.

In [29]:
import asyncio
import pandas as pd
import os
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DEFAULT_OUTPUT = "scraped_jobs.csv"

async def scrape_indeed(keyword, location, output_file=DEFAULT_OUTPUT, headless=False):
    """
    Busca ofertas en Indeed España y las añade a un CSV unificado (sin duplicados).

    Parámetros
    ----------
    keyword : str
    location : str
    output_file : str
        Nombre del archivo CSV en data/raw/ (por defecto 'scraped_jobs.csv').
    headless : bool

    Retorna
    -------
    pd.DataFrame con las ofertas únicas acumuladas (nuevas + existentes).
    """
    base_url = "https://es.indeed.com/jobs"
    query = f"q={keyword.replace(' ', '+')}&l={location.replace(' ', '+')}"
    url = f"{base_url}?{query}"

    print(f"Buscando: {keyword} en {location}")
    print(f"URL: {url}")

    async with async_playwright() as p:
        navegador = await p.chromium.launch(headless=headless)
        pagina = await navegador.new_page()
        await pagina.goto(url, wait_until="load", timeout=60000)
        
        try:
            await pagina.wait_for_selector(".job_seen_beacon", state="visible", timeout=15000)
            print("Tarjetas visibles detectadas.")
        except:
            print("Advertencia: no se detectaron tarjetas con .job_seen_beacon, continuando...")
        
        await asyncio.sleep(2)
        html = await pagina.content()
        await navegador.close()

    soup = BeautifulSoup(html, "html.parser")
    tarjetas = soup.select(".job_seen_beacon, .resultContent, .jobCard, div[data-jk]")
    print(f"Tarjetas parseadas: {len(tarjetas)}")

    nuevas_ofertas = []
    for tarjeta in tarjetas:
        titulo_elem = tarjeta.select_one("a.jcs-JobTitle span[id^='jobTitle']")
        titulo = titulo_elem.get_text(strip=True) if titulo_elem else None

        empresa_elem = tarjeta.select_one('[data-testid="company-name"]')
        empresa = empresa_elem.get_text(strip=True) if empresa_elem else None

        ubicacion_elem = tarjeta.select_one('[data-testid="text-location"]')
        ubicacion = ubicacion_elem.get_text(strip=True) if ubicacion_elem else None

        enlace_elem = tarjeta.select_one("a[data-jk]")
        enlace = None
        if enlace_elem:
            href = enlace_elem.get("href")
            if href:
                enlace = "https://es.indeed.com" + href if href.startswith("/") else href

        metadatos_elems = tarjeta.select('[data-testid="attribute_snippet_testid"]')
        metadatos = [m.get_text(strip=True) for m in metadatos_elems]
        metadatos_str = " | ".join(metadatos) if metadatos else None

        nuevas_ofertas.append({
            "titulo": titulo,
            "empresa": empresa,
            "ubicacion": ubicacion,
            "enlace": enlace,
            "metadatos": metadatos_str
        })

    df_nuevas = pd.DataFrame(nuevas_ofertas)
    print(f"Nuevas ofertas extraídas: {len(df_nuevas)}")

    # --- Gestión del archivo unificado ---
    raw_dir = os.path.join(PROJECT_ROOT, "data", "raw")
    os.makedirs(raw_dir, exist_ok=True)
    filepath = os.path.join(raw_dir, output_file)

    # Cargar ofertas existentes si el archivo ya existe
    if os.path.exists(filepath):
        df_existente = pd.read_csv(filepath)
        print(f"Ofertas ya existentes en {output_file}: {len(df_existente)}")
        # Combinar y eliminar duplicados por 'enlace' (clave única)
        df_total = pd.concat([df_existente, df_nuevas], ignore_index=True)
        df_total = df_total.drop_duplicates(subset=["enlace"], keep="first")
    else:
        df_total = df_nuevas

    df_total.to_csv(filepath, index=False, encoding="utf-8")
    print(f"Total de ofertas únicas guardadas en {filepath}: {len(df_total)}")

    return df_total

In [30]:
# Probar la función con "analista de datos" en Madrid
df = await scrape_indeed("analista de datos", "Madrid", headless=False)

# Mostrar las primeras filas
df.head()

Buscando: analista de datos en Madrid
URL: https://es.indeed.com/jobs?q=analista+de+datos&l=Madrid
Tarjetas visibles detectadas.
Tarjetas parseadas: 32
Nuevas ofertas extraídas: 32
Total de ofertas únicas guardadas en /mnt/hdd/David/2026/Somos_F5/Bootcam_Desarrollo_IA/Proyectos/Modulo2/1_Proyecto/proyecto-eda-roles-datos/data/raw/scraped_jobs.csv: 32


,titulo,empresa,ubicacion,enlace,metadatos
0,DPO Analista de Datos,SEK EDUCATION GROUP,"Villanueva de la Cañada, Madrid provincia",https://es.indeed.com/rc/clk?jk=6e24f34f410689...,Jornada completa+1
1,DPO Analista de Datos,SEK EDUCATION GROUP,"Villanueva de la Cañada, Madrid provincia",https://es.indeed.com/rc/clk?jk=6e24f34f410689...,Jornada completa+1
2,Analista de datos,domestiko.com,"Madrid, Madrid provincia",https://es.indeed.com/rc/clk?jk=f9672f50d90f3b...,Jornada completa+1 | De lunes a viernes
3,Analista de datos,domestiko.com,"Madrid, Madrid provincia",https://es.indeed.com/rc/clk?jk=f9672f50d90f3b...,Jornada completa+1 | De lunes a viernes
4,27762 - Analista de Datos,INECO,"28036 Madrid, Madrid provincia",https://es.indeed.com/rc/clk?jk=687fab71451875...,NaN


In [31]:
# Búsqueda múltiple de diferentes perfiles de datos en España
busquedas = [
    ("data scientist", "Madrid"),
    ("data engineer", "Barcelona"),
    ("machine learning", "Valencia"),
    # Puedes añadir más tuplas (keyword, location) aquí
]

for keyword, location in busquedas:
    print(f"\n{'='*50}")
    print(f"Iniciando búsqueda: {keyword} en {location}")
    await scrape_indeed(keyword, location, headless=False)
    print(f"Búsqueda de {keyword} completada.\n")

print("Todas las búsquedas finalizadas. Los datos están en data/raw/scraped_jobs.csv")


Iniciando búsqueda: data scientist en Madrid
Buscando: data scientist en Madrid
URL: https://es.indeed.com/jobs?q=data+scientist&l=Madrid
Tarjetas visibles detectadas.
Tarjetas parseadas: 32
Nuevas ofertas extraídas: 32
Ofertas ya existentes en scraped_jobs.csv: 32
Total de ofertas únicas guardadas en /mnt/hdd/David/2026/Somos_F5/Bootcam_Desarrollo_IA/Proyectos/Modulo2/1_Proyecto/proyecto-eda-roles-datos/data/raw/scraped_jobs.csv: 32
Búsqueda de data scientist completada.


Iniciando búsqueda: data engineer en Barcelona
Buscando: data engineer en Barcelona
URL: https://es.indeed.com/jobs?q=data+engineer&l=Barcelona
Tarjetas visibles detectadas.
Tarjetas parseadas: 32
Nuevas ofertas extraídas: 32
Ofertas ya existentes en scraped_jobs.csv: 32
Total de ofertas únicas guardadas en /mnt/hdd/David/2026/Somos_F5/Bootcam_Desarrollo_IA/Proyectos/Modulo2/1_Proyecto/proyecto-eda-roles-datos/data/raw/scraped_jobs.csv: 48
Búsqueda de data engineer completada.


Iniciando búsqueda: machine learning